In [36]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import json
import pandas as pd
import time
import torch

In [37]:
print('torch version:', torch.__version__)
print('CUDA version', torch.version.cuda)
print('CUDA is available:', torch.cuda.is_available())
print('Number of GPUs available:', torch.cuda.device_count())

torch version: 2.3.0
CUDA version 12.1
CUDA is available: True
Number of GPUs available: 4


In [ ]:
file_path = ""
results_filename = "results.csv"
latency_filename = "latency_results.csv"

In [39]:
def export_model(file_name, data, headers):
    with open(file_name, mode='w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        writer.writerow(headers)  # Write the header row
        writer.writerows(data)  # Write all rows of data

In [40]:
models = [
    #"Qwen/Qwen2.5-7B-Instruct-1M",
    "Qwen/Qwen2.5-14B-Instruct-1M"
]

In [41]:
prompt_instructions = """
You are an AI driving analysis tool. Your primary function is to assess whether a given driving scenario is consistent or exhibits signs of an adversarial attack (e.g., tampered traffic signs). You have expert knowledge of traffic regulations, road signage, and common driving practices.

Given a driving scenario in JSON format, your goal is to:
    1. Determine if there is any inconsistency in the scenario (e.g., a sign or speed limit that conflicts with typical real-world traffic laws).
    2. Provide a clear and concise explanation of your reasoning, identifying which specific part of the scenario is inconsistent and referencing relevant traffic rules or logical deductions.

Use the following exact response format. Fill in the placeholders appropriately:

    Inconsistency Detected: [Yes/No]
    Reasoning: [Provide a thorough and concise explanation. Keep it to 3-5 sentences and ensure it demonstrates your reasoning.]

Follow a step-by-step reasoning process internally but only provide the final output in the specified format. Be precise and adhere strictly to the format.
"""

In [42]:
def load_scenarios(file_path):
    with open(file_path, "r") as file:
        scenarios = json.load(file)
    return scenarios["Scenarios"]

In [43]:
scenario_filepath = file_path + "scenarios.json"
scenarios = load_scenarios(scenario_filepath)

In [44]:
for model_name in models:

    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype="auto",
        device_map="auto"
    )

    tokenizer = AutoTokenizer.from_pretrained(model_name)

    outputs = []
    latency_outputs = []

    for scenario in scenarios:
        scenario_input =(
            f"Scenario ID: {scenario['id']}\n"
            f"Road Type: {scenario['RoadType']}\n"
            f"Speed: {scenario['Speed']} mph\n"
            f"Lane Markings: {', '.join(scenario['LaneMarkings'])}\n"
            f"Traffic Signs: {', '.join(scenario['TrafficSigns'])}\n"
            f"{', '.join([str(device) for device in scenario['TrafficControlDevices']]) if scenario['TrafficControlDevices'] else 'None'}\n"
            f"Time of Day: {scenario['TimeOfDay']}\n"
            f"Description: {scenario['Description']}\n"
        )
        print(f"******************************************************************************************")
        print(f"For Model: {model_name} - Processing Scenario {scenario['id']}...")
        start_time = time.time()

        messages = [
            {"role": "system", "content": prompt_instructions},
            {"role": "user", "content": scenario_input}
        ]

        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=512
        )

        elapsed_time = round(time.time() - start_time, 4)

        gnerated_ids = [
            output_ids[len(input_ids):] for input_ids, output_ids in zip(model_inputs, generated_ids)
        ]
        
        generated_text = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        outputs.append(generated_text)
        latency_outputs.append(elapsed_time)
    
    ## Output results
    # read current csv file 
    current_results_df = pd.read_csv(results_filename)
    scenario_id = "scenario_id"
    
    new_results_df = pd.DataFrame({
        scenario_id: [scenario["id"] for scenario in scenarios],
        model_name: outputs
    })

    # Merge new model's ouputs with existing DataFrames
    updated_df = pd.merge(current_results_df, new_results_df, on=scenario_id, how="left")

    updated_df.to_csv(results_filename, index=False)

    
    ## Output Latency results
    #read current latency csv file 
    current_latency_results_df = pd.read_csv(latency_filename)

    new_latency_df = pd.DataFrame({
        scenario_id: [scenario["id"] for scenario in scenarios],
        f"latency_{model_name}": latency_outputs

    })

    # Merge new model's ouputs with existing DataFrames
    updated_latency_df = pd.merge(current_latency_results_df, new_latency_df, on=scenario_id, how="left")

    updated_latency_df.to_csv(latency_filename, index=False)

    del model
    del tokenizer


Loading checkpoint shards:  25%|██▌       | 2/8 [00:20<01:02, 10.46s/it]

******************************************************************************************
For Model: Qwen/Qwen2.5-14B-Instruct-1M - Processing Scenario 13...
******************************************************************************************
For Model: Qwen/Qwen2.5-14B-Instruct-1M - Processing Scenario 14...
******************************************************************************************
For Model: Qwen/Qwen2.5-14B-Instruct-1M - Processing Scenario 15...
******************************************************************************************
For Model: Qwen/Qwen2.5-14B-Instruct-1M - Processing Scenario 16...
******************************************************************************************
For Model: Qwen/Qwen2.5-14B-Instruct-1M - Processing Scenario 17...
******************************************************************************************
For Model: Qwen/Qwen2.5-14B-Instruct-1M - Processing Scenario 18...
**********************************************